<a href="https://colab.research.google.com/github/AASTHASHENKAR/SAT_CEP_Monitor_Big_Data_Analytics/blob/main/SAT_CEP_Monitor_BDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️ SAT-CEP-Monitor: Air Pollution Event Detector
## Big Data Analytics (BDA) Mini Project

**Based on:** *SAT-CEP-Monitor: An air quality monitoring software architecture combining complex event processing with satellite remote sensing*

---

### 📐 Architecture Overview
```
Satellite Remote Sensing Data (AOD, NO2, SO2, CO, O3)
      ↓
[Data Ingestion Layer]     ← PySpark / Simulated Data Streams
      ↓
[Preprocessing Layer]      ← Cleaning, Validation, Feature Engineering
      ↓
[Big Data Processing]      ← PySpark RDD + DataFrame Transformations
      ↓
[CEP Engine]               ← Complex Event Processing Rules (8 Rules)
      ↓
[Pollution Event Detector] ← Threshold + Pattern-Based Alerts
      ↓
[Analytics & Reporting]    ← AQI Computation, Trend Analysis, Rankings
```

### 🗺️ Regions Covered
- **Morocco:** Casablanca, Rabat, Marrakech, Fes
- **Spain:** Madrid, Barcelona, Seville, Valencia

### 🛰️ Satellite Sensors Simulated
MODIS-Terra | MODIS-Aqua | Sentinel-5P | VIIRS | TROPOMI

 Install PySpark

In [ ]:
# Install PySpark (only needed in Colab)
!pip install pyspark -q
print("✔ PySpark installed successfully")

✔ PySpark installed successfully


 Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, FloatType,
    IntegerType, DoubleType
)
from pyspark.sql.window import Window
import random
import datetime

print("✔ All imports successful")

✔ All imports successful


Initialize Spark Session

In [ ]:
spark = SparkSession.builder \
    .appName("SAT-CEP-Monitor: Air Pollution Event Detector") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("=" * 60)
print("  SAT-CEP-Monitor: Air Pollution Event Detector")
print("  PySpark Version:", spark.version)
print("=" * 60)

  SAT-CEP-Monitor: Air Pollution Event Detector
  PySpark Version: 4.0.2


 Schema Definition

Defines the structure of satellite remote sensing data including:
- **AOD** — Aerosol Optical Depth (how much particles block sunlight)
- **NO₂, SO₂, CO, O₃** — Trace gases measured via spectroscopy
- **PM2.5, PM10** — Estimated from AOD using physical models

In [ ]:
satellite_schema = StructType([
    StructField("reading_id",     IntegerType(), True),
    StructField("timestamp",      StringType(),  True),
    StructField("region",         StringType(),  True),
    StructField("country",        StringType(),  True),
    StructField("latitude",       DoubleType(),  True),
    StructField("longitude",      DoubleType(),  True),
    StructField("sensor",         StringType(),  True),
    # --- Atmospheric Parameters ---
    StructField("AOD",            FloatType(),   True),  # Aerosol Optical Depth
    StructField("NO2",            FloatType(),   True),  # µmol/m²
    StructField("SO2",            FloatType(),   True),  # µmol/m²
    StructField("CO",             FloatType(),   True),  # mol/m²
    StructField("O3",             FloatType(),   True),  # Dobson Units
    StructField("PM25_estimated", FloatType(),   True),  # µg/m³ (derived from AOD)
    StructField("PM10_estimated", FloatType(),   True),  # µg/m³ (derived from AOD)
    StructField("temp_celsius",   FloatType(),   True),
    StructField("humidity_pct",   FloatType(),   True),
    StructField("wind_speed_ms",  FloatType(),   True),
    StructField("cloud_cover",    FloatType(),   True),
])

print("✔ Schema defined with", len(satellite_schema.fields), "fields")

✔ Schema defined with 18 fields


 Data Generation

Simulates satellite remote sensing data for **Morocco and Spain** (exact regions from the SAT-CEP-Monitor paper). Pollution events are injected probabilistically to test CEP detection.

In [ ]:
def generate_satellite_data(n_records=500):
    """
    Simulate satellite remote sensing data for Morocco and Spain.
    Includes normal background readings + injected pollution events.
    """
    random.seed(42)

    regions = {
        "Casablanca": ("Morocco", 33.57, -7.58),
        "Rabat":      ("Morocco", 34.02, -6.83),
        "Marrakech":  ("Morocco", 31.62, -7.98),
        "Fes":        ("Morocco", 34.03, -5.00),
        "Madrid":     ("Spain",   40.42, -3.70),
        "Barcelona":  ("Spain",   41.38,  2.17),
        "Seville":    ("Spain",   37.38, -5.99),
        "Valencia":   ("Spain",   39.47, -0.37),
    }

    sensors = ["MODIS-Terra", "MODIS-Aqua", "Sentinel-5P", "VIIRS", "TROPOMI"]

    # Pollution hotspot probabilities per region
    hotspots = {
        "Casablanca": 0.35,  # High industrial activity
        "Madrid":     0.30,
        "Barcelona":  0.25,
        "Marrakech":  0.20,
    }

    base_time = datetime.datetime(2024, 1, 1, 0, 0, 0)
    records = []

    for i in range(n_records):
        region = random.choice(list(regions.keys()))
        country, lat, lon = regions[region]
        sensor = random.choice(sensors)
        ts = base_time + datetime.timedelta(hours=i * 0.5)
        ts_str = ts.strftime("%Y-%m-%d %H:%M:%S")

        is_event = random.random() < hotspots.get(region, 0.10)

        if is_event:
            aod  = round(random.uniform(0.6,  1.5),  3)
            no2  = round(random.uniform(150,  450),  2)
            so2  = round(random.uniform(50,   200),  2)
            co   = round(random.uniform(0.08, 0.20), 4)
            o3   = round(random.uniform(320,  420),  2)
            pm25 = round(aod * random.uniform(60, 90), 2)
            pm10 = round(pm25 * random.uniform(1.5, 2.2), 2)
        else:
            aod  = round(random.uniform(0.02, 0.35), 3)
            no2  = round(random.uniform(10,   80),   2)
            so2  = round(random.uniform(2,    30),   2)
            co   = round(random.uniform(0.01, 0.05), 4)
            o3   = round(random.uniform(220,  300),  2)
            pm25 = round(aod * random.uniform(20, 45), 2)
            pm10 = round(pm25 * random.uniform(1.3, 1.9), 2)

        temp     = round(random.uniform(5,  42), 1)
        humidity = round(random.uniform(20, 90), 1)
        wind     = round(random.uniform(0,  15), 2)
        cloud    = round(random.uniform(0,  1),  2)

        records.append((
            i + 1, ts_str, region, country, lat, lon, sensor,
            aod, no2, so2, co, o3, pm25, pm10,
            temp, humidity, wind, cloud
        ))

    return records

print("✔ Data generator function defined")

✔ Data generator function defined


 STEP 1: Data Ingestion

In [ ]:
print("[STEP 1] Ingesting satellite remote sensing data stream...")
print("-" * 55)

raw_data = generate_satellite_data(500)
raw_df   = spark.createDataFrame(raw_data, schema=satellite_schema)
raw_df   = raw_df.withColumn("timestamp", F.to_timestamp("timestamp"))

total_records = raw_df.count()

print(f"  Total records ingested  : {total_records}")
print(f"  Regions covered         : {raw_df.select('region').distinct().count()}")
print(f"  Satellite sensors       : {raw_df.select('sensor').distinct().count()}")
print(f"  Countries               : Morocco, Spain")
print()
print("Sample Data:")
raw_df.select("timestamp", "region", "country", "sensor", "AOD", "NO2", "PM25_estimated").show(5, truncate=False)

[STEP 1] Ingesting satellite remote sensing data stream...
-------------------------------------------------------
  Total records ingested  : 500
  Regions covered         : 8
  Satellite sensors       : 5
  Countries               : Morocco, Spain

Sample Data:
+-------------------+---------+-------+-----------+-----+-----+--------------+
|timestamp          |region   |country|sensor     |AOD  |NO2  |PM25_estimated|
+-------------------+---------+-------+-----------+-----+-----+--------------+
|2024-01-01 00:00:00|Rabat    |Morocco|MODIS-Terra|0.101|19.77|3.51          |
|2024-01-01 00:30:00|Seville  |Spain  |MODIS-Aqua |0.112|70.85|3.02          |
|2024-01-01 01:00:00|Barcelona|Spain  |TROPOMI    |0.034|42.16|0.93          |
|2024-01-01 01:30:00|Fes      |Morocco|Sentinel-5P|0.302|70.65|7.27          |
|2024-01-01 02:00:00|Marrakech|Morocco|TROPOMI    |0.074|36.56|2.75          |
+-------------------+---------+-------+-----------+-----+-----+--------------+
only showing top 5 rows


 STEP 2: Preprocessing & Validation

In [ ]:
print("[STEP 2] Preprocessing & Data Validation...")
print("-" * 55)

# Drop nulls in critical columns
cleaned_df = raw_df.dropna(subset=["AOD", "NO2", "SO2", "CO", "O3",
                                    "PM25_estimated", "PM10_estimated"])

# Filter physically impossible values
cleaned_df = cleaned_df.filter(
    (F.col("AOD")  >= 0) & (F.col("AOD")  <= 5.0) &
    (F.col("NO2")  >= 0) & (F.col("NO2")  <= 1000) &
    (F.col("SO2")  >= 0) & (F.col("SO2")  <= 500) &
    (F.col("CO")   >= 0) & (F.col("CO")   <= 1.0) &
    (F.col("O3")   >= 0) & (F.col("O3")   <= 600) &
    (F.col("PM25_estimated") >= 0) &
    (F.col("PM10_estimated") >= 0)
)

removed = total_records - cleaned_df.count()
print(f"  Records after cleaning  : {cleaned_df.count()} ({removed} removed as invalid)")
print(f"  Null check              : {cleaned_df.filter(F.col('AOD').isNull()).count()} nulls in AOD")
print()

# Basic statistics
print("Descriptive Statistics:")
cleaned_df.select("AOD", "NO2", "SO2", "CO", "PM25_estimated", "PM10_estimated").describe().show()

[STEP 2] Preprocessing & Data Validation...
-------------------------------------------------------
  Records after cleaning  : 500 (0 removed as invalid)
  Null check              : 0 nulls in AOD

Descriptive Statistics:
+-------+------------------+------------------+-----------------+-------------------+------------------+-----------------+
|summary|               AOD|               NO2|              SO2|                 CO|    PM25_estimated|   PM10_estimated|
+-------+------------------+------------------+-----------------+-------------------+------------------+-----------------+
|  count|               500|               500|              500|                500|               500|              500|
|   mean|0.3346440000385046|  87.1790402584076|35.20156003522873|0.04829600005596876|18.607900012016298| 33.2932799924612|
| stddev| 0.354110085271698|103.33772146295362|46.86509405773763|0.04549037399422791|29.394177890062764|55.92567280522225|
|    min|             0.024|           

 STEP 3: AQI Computation (US EPA Standard)

Since satellites cannot directly measure PM2.5, it is **derived from AOD** using physical models. AQI is then computed from the estimated PM2.5 value.

In [ ]:
print("[STEP 3] Computing Air Quality Index (AQI)...")
print("-" * 55)

def compute_aqi_pm25(pm25_col):
    """US EPA AQI breakpoints for PM2.5 (µg/m³)"""
    return (
        F.when(pm25_col <= 12.0,  F.round((50 / 12.0) * pm25_col, 1))
         .when(pm25_col <= 35.4,  F.round(((100-51)  / (35.4-12.1)) * (pm25_col-12.1) + 51, 1))
         .when(pm25_col <= 55.4,  F.round(((150-101) / (55.4-35.5)) * (pm25_col-35.5) + 101, 1))
         .when(pm25_col <= 150.4, F.round(((200-151) / (150.4-55.5))* (pm25_col-55.5) + 151, 1))
         .when(pm25_col <= 250.4, F.round(((300-201) / (250.4-150.5))*(pm25_col-150.5)+ 201, 1))
         .otherwise(F.round(((500-301)/(500.4-250.5))*(pm25_col-250.5)+301, 1))
    )

def aqi_category(aqi_col):
    return (
        F.when(aqi_col <= 50,  "Good")
         .when(aqi_col <= 100, "Moderate")
         .when(aqi_col <= 150, "Unhealthy for Sensitive Groups")
         .when(aqi_col <= 200, "Unhealthy")
         .when(aqi_col <= 300, "Very Unhealthy")
         .otherwise("Hazardous")
    )

enriched_df = cleaned_df \
    .withColumn("AQI",          compute_aqi_pm25(F.col("PM25_estimated"))) \
    .withColumn("AQI_category", aqi_category(F.col("AQI")))

print("AQI Category Distribution:")
enriched_df.groupBy("AQI_category") \
           .count() \
           .withColumn("percentage", F.round(F.col("count") * 100.0 / total_records, 1)) \
           .orderBy("count", ascending=False) \
           .show(truncate=False)

[STEP 3] Computing Air Quality Index (AQI)...
-------------------------------------------------------
AQI Category Distribution:
+------------------------------+-----+----------+
|AQI_category                  |count|percentage|
+------------------------------+-----+----------+
|Good                          |395  |79.0      |
|Unhealthy                     |72   |14.4      |
|Moderate                      |22   |4.4       |
|Unhealthy for Sensitive Groups|11   |2.2       |
+------------------------------+-----+----------+



 STEP 4: CEP Engine (Complex Event Processing)

The **core of SAT-CEP-Monitor**. Implements 8 CEP rules based on WHO/EU air quality guidelines:

| Rule | Condition | Threshold |
|------|-----------|----------|
| AOD Spike | AOD ≥ 0.6 | High aerosol load |
| NO₂ Elevated | NO₂ ≥ 100 µmol/m² | Trace gas alert |
| SO₂ Industrial | SO₂ ≥ 40 µmol/m² | Industrial/volcanic |
| CO Combustion | CO ≥ 0.07 mol/m² | Combustion source |
| PM2.5 Moderate | PM2.5 ≥ 35.4 µg/m³ | WHO threshold |
| PM2.5 Unhealthy | PM2.5 ≥ 55.4 µg/m³ | EU limit |
| **Compound Event** | AOD + NO₂ + PM2.5 all exceed | Multi-pollutant |
| Severe Alert | AQI ≥ 150 | Unhealthy level |

In [ ]:
print("[STEP 4] Running Complex Event Processing (CEP) Engine...")
print("-" * 55)

# --- CEP Thresholds ---
AOD_HIGH       = 0.6
NO2_HIGH       = 100.0
SO2_HIGH       = 40.0
CO_HIGH        = 0.07
PM25_MODERATE  = 35.4
PM25_UNHEALTHY = 55.4
PM10_HIGH      = 75.0
AQI_ALERT      = 100
AQI_CRITICAL   = 150

# Apply all 8 CEP rules
cep_df = enriched_df \
    .withColumn("CEP_AOD_spike",
        F.when(F.col("AOD") >= AOD_HIGH, True).otherwise(False)) \
    .withColumn("CEP_NO2_elevated",
        F.when(F.col("NO2") >= NO2_HIGH, True).otherwise(False)) \
    .withColumn("CEP_SO2_industrial",
        F.when(F.col("SO2") >= SO2_HIGH, True).otherwise(False)) \
    .withColumn("CEP_CO_combustion",
        F.when(F.col("CO") >= CO_HIGH, True).otherwise(False)) \
    .withColumn("CEP_PM25_moderate",
        F.when(F.col("PM25_estimated") >= PM25_MODERATE, True).otherwise(False)) \
    .withColumn("CEP_PM25_unhealthy",
        F.when(F.col("PM25_estimated") >= PM25_UNHEALTHY, True).otherwise(False)) \
    .withColumn("CEP_compound_event",
        F.when(
            (F.col("AOD") >= AOD_HIGH) &
            (F.col("NO2") >= NO2_HIGH) &
            (F.col("PM25_estimated") >= PM25_MODERATE), True
        ).otherwise(False)) \
    .withColumn("CEP_severe_alert",
        F.when(F.col("AQI") >= AQI_CRITICAL, True).otherwise(False))

# Count rules fired per reading
event_flags = [
    "CEP_AOD_spike", "CEP_NO2_elevated", "CEP_SO2_industrial",
    "CEP_CO_combustion", "CEP_PM25_moderate", "CEP_PM25_unhealthy",
    "CEP_compound_event", "CEP_severe_alert"
]

cep_df = cep_df.withColumn(
    "CEP_rules_fired",
    sum(F.col(c).cast(IntegerType()) for c in event_flags)
)

# Severity classification
cep_df = cep_df.withColumn(
    "pollution_severity",
    F.when(F.col("CEP_rules_fired") == 0, "NORMAL")
     .when(F.col("CEP_rules_fired") == 1, "LOW_ALERT")
     .when(F.col("CEP_rules_fired") == 2, "MODERATE_ALERT")
     .when(F.col("CEP_rules_fired") == 3, "HIGH_ALERT")
     .otherwise("CRITICAL_EVENT")
)

cep_df.cache()
print("✔ CEP Engine complete. 8 rules applied.")

[STEP 4] Running Complex Event Processing (CEP) Engine...
-------------------------------------------------------
✔ CEP Engine complete. 8 rules applied.


 STEP 5: Pollution Event Detection Results

In [ ]:
print("[STEP 5] Pollution Event Detection Results")
print("-" * 55)

total = cep_df.count()

print("\n📊 Pollution Severity Distribution:")
cep_df.groupBy("pollution_severity") \
      .count() \
      .withColumn("percentage", F.round(F.col("count") * 100.0 / total, 2)) \
      .orderBy("count", ascending=False) \
      .show(truncate=False)

total_events = cep_df.filter(F.col("pollution_severity") != "NORMAL").count()
print(f"Total pollution events detected : {total_events} / {total} readings")
print(f"Event detection rate            : {round(total_events*100/total, 2)}%")

[STEP 5] Pollution Event Detection Results
-------------------------------------------------------

📊 Pollution Severity Distribution:
+------------------+-----+----------+
|pollution_severity|count|percentage|
+------------------+-----+----------+
|NORMAL            |417  |83.4      |
|CRITICAL_EVENT    |83   |16.6      |
+------------------+-----+----------+

Total pollution events detected : 83 / 500 readings
Event detection rate            : 16.6%


STEP 6: CEP Rule Activation Summary

In [ ]:
print("[STEP 6] CEP Rule Activation Summary")
print("-" * 55)

rule_data = []
for flag in event_flags:
    count = cep_df.filter(F.col(flag) == True).count()
    pct   = round(count * 100 / total, 2)
    label = flag.replace("CEP_", "").replace("_", " ").title()
    rule_data.append((label, count, pct))

rule_schema = StructType([
    StructField("CEP_Rule",    StringType(),  True),
    StructField("Activations", IntegerType(), True),
    StructField("Rate_Pct",    FloatType(),   True),
])
rule_df = spark.createDataFrame(rule_data, schema=rule_schema)
rule_df.orderBy("Activations", ascending=False).show(truncate=False)

[STEP 6] CEP Rule Activation Summary
-------------------------------------------------------
+--------------+-----------+--------+
|CEP_Rule      |Activations|Rate_Pct|
+--------------+-----------+--------+
|Pm25 Moderate |83         |16.6    |
|Aod Spike     |83         |16.6    |
|Compound Event|83         |16.6    |
|No2 Elevated  |83         |16.6    |
|So2 Industrial|83         |16.6    |
|Co Combustion |83         |16.6    |
|Pm25 Unhealthy|72         |14.4    |
|Severe Alert  |72         |14.4    |
+--------------+-----------+--------+



 STEP 7: Regional Air Quality Analysis

In [ ]:
print("[STEP 7] Regional Air Quality Analysis")
print("-" * 55)

regional_stats = cep_df.groupBy("region", "country") \
    .agg(
        F.count("*").alias("total_readings"),
        F.round(F.avg("AOD"),            4).alias("avg_AOD"),
        F.round(F.avg("NO2"),            2).alias("avg_NO2"),
        F.round(F.avg("SO2"),            2).alias("avg_SO2"),
        F.round(F.avg("PM25_estimated"), 2).alias("avg_PM25"),
        F.round(F.avg("AQI"),            2).alias("avg_AQI"),
        F.round(F.max("AQI"),            2).alias("max_AQI"),
        F.sum(F.col("CEP_compound_event").cast(IntegerType())).alias("compound_events"),
        F.sum(F.col("CEP_severe_alert").cast(IntegerType())).alias("severe_alerts"),
    ).orderBy("avg_AQI", ascending=False)

regional_stats.show(truncate=False)

[STEP 7] Regional Air Quality Analysis
-------------------------------------------------------
+----------+-------+--------------+-------+-------+-------+--------+-------+-------+---------------+-------------+
|region    |country|total_readings|avg_AOD|avg_NO2|avg_SO2|avg_PM25|avg_AQI|max_AQI|compound_events|severe_alerts|
+----------+-------+--------------+-------+-------+-------+--------+-------+-------+---------------+-------------+
|Madrid    |Spain  |65            |0.436  |114.68 |50.13  |27.74   |65.77  |187.8  |19             |16           |
|Marrakech |Morocco|68            |0.417  |107.92 |38.93  |25.85   |59.45  |186.1  |16             |15           |
|Casablanca|Morocco|69            |0.3997 |114.39 |48.37  |23.98   |59.16  |184.9  |18             |14           |
|Barcelona |Spain  |64            |0.3833 |104.98 |44.87  |22.88   |57.15  |186.4  |15             |12           |
|Seville   |Spain  |56            |0.2772 |69.62  |25.06  |13.86   |39.52  |183.3  |5              |

 STEP 8: Top 10 Critical Events

In [ ]:
print("[STEP 8] Top 10 Most Critical Pollution Events Detected")
print("-" * 55)

cep_df \
    .filter(F.col("pollution_severity").isin(["CRITICAL_EVENT", "HIGH_ALERT"])) \
    .select("timestamp", "region", "country", "sensor",
            "AOD", "NO2", "PM25_estimated", "AQI",
            "AQI_category", "CEP_rules_fired", "pollution_severity") \
    .orderBy("CEP_rules_fired", "AQI", ascending=False) \
    .limit(10) \
    .show(truncate=False)

[STEP 8] Top 10 Most Critical Pollution Events Detected
-------------------------------------------------------
+-------------------+----------+-------+-----------+-----+------+--------------+-----+------------+---------------+------------------+
|timestamp          |region    |country|sensor     |AOD  |NO2   |PM25_estimated|AQI  |AQI_category|CEP_rules_fired|pollution_severity|
+-------------------+----------+-------+-----------+-----+------+--------------+-----+------------+---------------+------------------+
|2024-01-01 02:30:00|Madrid    |Spain  |MODIS-Terra|1.449|412.91|126.7         |187.8|Unhealthy   |8              |CRITICAL_EVENT    |
|2024-01-01 04:30:00|Barcelona |Spain  |VIIRS      |1.436|413.62|123.98        |186.4|Unhealthy   |8              |CRITICAL_EVENT    |
|2024-01-03 23:30:00|Marrakech |Morocco|Sentinel-5P|1.494|267.92|123.53        |186.1|Unhealthy   |8              |CRITICAL_EVENT    |
|2024-01-04 07:00:00|Casablanca|Morocco|MODIS-Aqua |1.381|416.27|121.21       

 Temporal Trend Analysis (Hourly)

In [ ]:
print("[STEP 9] Temporal Trend Analysis — Hourly Aggregation")
print("-" * 55)

temporal_df = cep_df \
    .withColumn("hour", F.hour("timestamp")) \
    .withColumn("date", F.to_date("timestamp"))

hourly_trend = temporal_df.groupBy("hour") \
    .agg(
        F.round(F.avg("AQI"),             2).alias("avg_AQI"),
        F.round(F.avg("AOD"),             4).alias("avg_AOD"),
        F.round(F.avg("PM25_estimated"),  2).alias("avg_PM25"),
        F.sum(F.col("CEP_severe_alert").cast(IntegerType())).alias("severe_alerts"),
    ).orderBy("hour")

hourly_trend.show(24, truncate=False)

[STEP 9] Temporal Trend Analysis — Hourly Aggregation
-------------------------------------------------------
+----+-------+-------+--------+-------------+
|hour|avg_AQI|avg_AOD|avg_PM25|severe_alerts|
+----+-------+-------+--------+-------------+
|0   |35.2   |0.2479 |11.8    |2            |
|1   |34.37  |0.2485 |10.94   |2            |
|2   |54.21  |0.3798 |23.63   |3            |
|3   |29.27  |0.1844 |8.19    |1            |
|4   |60.14  |0.4027 |24.19   |4            |
|5   |51.42  |0.3543 |18.36   |3            |
|6   |44.82  |0.3062 |15.57   |2            |
|7   |84.46  |0.5967 |38.7    |7            |
|8   |46.4   |0.3418 |18.46   |3            |
|9   |42.25  |0.2887 |13.91   |2            |
|10  |43.43  |0.3103 |16.27   |2            |
|11  |48.25  |0.2956 |17.63   |3            |
|12  |61.71  |0.4195 |25.4    |5            |
|13  |49.35  |0.3369 |18.36   |3            |
|14  |36.35  |0.2827 |14.52   |2            |
|15  |45.79  |0.3177 |15.9    |3            |
|16  |23.58  |0.

 STEP 10: RDD-Based Processing

Demonstrates **low-level distributed computation** using PySpark RDD operations: `map`, `reduceByKey`, `filter`, `flatMap`.

In [ ]:
print("[STEP 10] RDD-Based Distributed Processing")
print("-" * 55)

raw_rdd = cep_df.select(
    "region", "AOD", "NO2", "SO2", "CO", "PM25_estimated", "AQI", "pollution_severity"
).rdd

# --- map + reduceByKey: Average AQI per region ---
aqi_pairs    = raw_rdd.map(lambda r: (r["region"], (float(r["AQI"]), 1)))
region_totals = aqi_pairs.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
region_avg   = region_totals.mapValues(lambda v: round(v[0]/v[1], 2))
sorted_avgs  = region_avg.sortBy(lambda x: x[1], ascending=False).collect()

print("\n📍 RDD map + reduceByKey — Average AQI per Region:")
for region, avg in sorted_avgs:
    bar = "▓" * min(int(avg / 15), 15)
    print(f"  {region:<13}  {avg:>7.2f}  {bar}")

# --- filter: count critical events ---
critical_rdd = raw_rdd.filter(
    lambda r: r["pollution_severity"] in ("CRITICAL_EVENT", "HIGH_ALERT")
)
print(f"\n🔴 RDD filter — Critical/High events: {critical_rdd.count()} records")

# --- flatMap + reduceByKey: pollutant flag frequency ---
def extract_flags(row):
    flags = []
    if float(row["AOD"]) >= AOD_HIGH:              flags.append("AOD_spike")
    if float(row["NO2"]) >= NO2_HIGH:              flags.append("NO2_elevated")
    if float(row["SO2"]) >= SO2_HIGH:              flags.append("SO2_industrial")
    if float(row["CO"])  >= CO_HIGH:               flags.append("CO_combustion")
    if float(row["PM25_estimated"]) >= PM25_MODERATE: flags.append("PM25_exceed")
    return [(f, 1) for f in flags]

flag_counts = raw_rdd.flatMap(extract_flags) \
                     .reduceByKey(lambda a, b: a + b) \
                     .sortBy(lambda x: x[1], ascending=False) \
                     .collect()

print("\n📊 RDD flatMap + reduceByKey — Pollutant Flag Frequency:")
for flag, cnt in flag_counts:
    print(f"  {flag:<20} : {cnt}")

[STEP 10] RDD-Based Distributed Processing
-------------------------------------------------------

📍 RDD map + reduceByKey — Average AQI per Region:
  Madrid           65.77  ▓▓▓▓
  Marrakech        59.45  ▓▓▓
  Casablanca       59.16  ▓▓▓
  Barcelona        57.15  ▓▓▓
  Seville          39.52  ▓▓
  Valencia         37.09  ▓▓
  Fes              34.41  ▓▓
  Rabat            28.16  ▓

🔴 RDD filter — Critical/High events: 83 records

📊 RDD flatMap + reduceByKey — Pollutant Flag Frequency:
  AOD_spike            : 83
  SO2_industrial       : 83
  PM25_exceed          : 83
  NO2_elevated         : 83
  CO_combustion        : 83


STEP 11: Window Functions (Rolling Trend Detection)

In [ ]:
print("[STEP 11] Window Functions — Rolling 5-Reading Average per Region")
print("-" * 55)

window_spec = Window.partitionBy("region") \
                    .orderBy("timestamp") \
                    .rowsBetween(-4, 0)

rolling_df = cep_df \
    .withColumn("rolling_avg_AQI",   F.round(F.avg("AQI").over(window_spec), 2)) \
    .withColumn("rolling_avg_AOD",   F.round(F.avg("AOD").over(window_spec), 4)) \
    .withColumn("rolling_max_PM25",  F.round(F.max("PM25_estimated").over(window_spec), 2))

print("\nSustained Pollution Periods (Rolling AQI ≥ 100) by Region:")
rolling_df \
    .filter(F.col("rolling_avg_AQI") >= AQI_ALERT) \
    .groupBy("region") \
    .agg(
        F.count("*").alias("sustained_periods"),
        F.round(F.max("rolling_avg_AQI"), 2).alias("peak_rolling_AQI"),
    ).orderBy("sustained_periods", ascending=False) \
    .show(truncate=False)

[STEP 11] Window Functions — Rolling 5-Reading Average per Region
-------------------------------------------------------

Sustained Pollution Periods (Rolling AQI ≥ 100) by Region:
+----------+-----------------+----------------+
|region    |sustained_periods|peak_rolling_AQI|
+----------+-----------------+----------------+
|Madrid    |15               |187.8           |
|Marrakech |8                |136.36          |
|Casablanca|7                |131.3           |
|Seville   |3                |113.74          |
+----------+-----------------+----------------+



 STEP 12: Satellite Sensor Performance Analysis

In [ ]:
print("[STEP 12] Satellite Sensor Performance Analysis")
print("-" * 55)

cep_df.groupBy("sensor") \
    .agg(
        F.count("*").alias("readings"),
        F.round(F.avg("AOD"), 4).alias("avg_AOD"),
        F.round(F.avg("AQI"), 2).alias("avg_AQI"),
        F.sum(F.col("CEP_compound_event").cast(IntegerType())).alias("compound_events"),
        F.round(F.avg("cloud_cover"), 3).alias("avg_cloud_cover"),
    ).orderBy("compound_events", ascending=False) \
    .show(truncate=False)

[STEP 12] Satellite Sensor Performance Analysis
-------------------------------------------------------
+-----------+--------+-------+-------+---------------+---------------+
|sensor     |readings|avg_AOD|avg_AQI|compound_events|avg_cloud_cover|
+-----------+--------+-------+-------+---------------+---------------+
|Sentinel-5P|95      |0.3749 |53.56  |19             |0.487          |
|MODIS-Terra|106     |0.3285 |48.21  |19             |0.488          |
|MODIS-Aqua |100     |0.3354 |49.67  |17             |0.468          |
|TROPOMI    |91      |0.34   |47.89  |15             |0.511          |
|VIIRS      |108     |0.3    |43.54  |13             |0.538          |
+-----------+--------+-------+-------+---------------+---------------+



 STEP 13: Ground Station Validation

The paper validates satellite-derived data against ground station measurements. This step compares satellite-estimated PM2.5 with simulated ground truth data.

In [ ]:
print("[STEP 13] Ground Station Validation (Satellite vs Ground)")
print("-" * 55)

random.seed(99)
ground_schema = StructType([
    StructField("region",         StringType(), True),
    StructField("ground_PM25",    FloatType(),  True),
    StructField("ground_AQI_cat", StringType(), True),
])

ground_data = [
    ("Casablanca", round(random.uniform(55, 120), 2), "Unhealthy"),
    ("Rabat",      round(random.uniform(20,  45), 2), "Moderate"),
    ("Marrakech",  round(random.uniform(30,  70), 2), "Unhealthy for Sensitive Groups"),
    ("Fes",        round(random.uniform(15,  40), 2), "Moderate"),
    ("Madrid",     round(random.uniform(40,  90), 2), "Unhealthy for Sensitive Groups"),
    ("Barcelona",  round(random.uniform(30,  65), 2), "Moderate"),
    ("Seville",    round(random.uniform(20,  50), 2), "Moderate"),
    ("Valencia",   round(random.uniform(15,  45), 2), "Good"),
]

ground_df  = spark.createDataFrame(ground_data, schema=ground_schema)
sat_avg    = cep_df.groupBy("region") \
                   .agg(F.round(F.avg("PM25_estimated"), 2).alias("satellite_PM25"))

validation_df = sat_avg.join(ground_df, on="region") \
    .withColumn("difference",
                F.round(F.abs(F.col("satellite_PM25") - F.col("ground_PM25")), 2)) \
    .withColumn("error_pct",
                F.round(F.col("difference") * 100.0 / F.col("ground_PM25"), 1)) \
    .orderBy("error_pct")

validation_df.show(truncate=False)

mape = validation_df.agg(F.round(F.avg("error_pct"), 2)).collect()[0][0]
print(f"Mean Absolute Percentage Error (MAPE): {mape}%")
print("✔ Validation consistent with SAT-CEP-Monitor paper's methodology")

[STEP 13] Ground Station Validation (Satellite vs Ground)
-------------------------------------------------------
+----------+--------------+-----------+------------------------------+----------+---------+
|region    |satellite_PM25|ground_PM25|ground_AQI_cat                |difference|error_pct|
+----------+--------------+-----------+------------------------------+----------+---------+
|Marrakech |25.85         |37.15      |Unhealthy for Sensitive Groups|11.3      |30.4     |
|Barcelona |22.88         |38.79      |Moderate                      |15.91     |41.0     |
|Fes       |11.68         |21.21      |Moderate                      |9.53      |44.9     |
|Seville   |13.86         |31.49      |Moderate                      |17.63     |56.0     |
|Madrid    |27.74         |77.99      |Unhealthy for Sensitive Groups|50.25     |64.4     |
|Valencia  |11.64         |35.53      |Good                          |23.89     |67.2     |
|Rabat     |7.44          |25.0       |Moderate           

 Final Dashboard Summary

In [ ]:
print("=" * 60)
print("  FINAL DASHBOARD — SAT-CEP-Monitor Results")
print("=" * 60)

total_alerts   = cep_df.filter(F.col("pollution_severity") != "NORMAL").count()
critical_count = cep_df.filter(F.col("pollution_severity") == "CRITICAL_EVENT").count()
high_count     = cep_df.filter(F.col("pollution_severity") == "HIGH_ALERT").count()
clean_count    = cep_df.filter(F.col("pollution_severity") == "NORMAL").count()
avg_aqi_global = round(cep_df.agg(F.avg("AQI")).collect()[0][0], 2)
compound_total = cep_df.filter(F.col("CEP_compound_event") == True).count()
worst_region   = regional_stats.orderBy("avg_AQI", ascending=False).first()["region"]
best_region    = regional_stats.orderBy("avg_AQI", ascending=True).first()["region"]

print(f"""
  Total Readings Processed     : {total}
  Clean / Normal Readings      : {clean_count}
  Total Pollution Events (CEP) : {total_alerts}
    ↳ Critical Events          : {critical_count}
    ↳ High Alert Events        : {high_count}
    ↳ Compound Multi-Pollutant : {compound_total}

  Global Average AQI           : {avg_aqi_global}
  Most Polluted Region         : {worst_region}
  Cleanest Region              : {best_region}

  Architecture Used:
    ✔  PySpark DataFrame API   (preprocessing, AQI computation)
    ✔  PySpark RDD API         (map, reduceByKey, filter, flatMap)
    ✔  Window Functions        (rolling trend / sustained pollution)
    ✔  CEP Engine              (8 rules, multi-parameter patterns)
    ✔  Ground Truth Validation (satellite vs ground station)
    ✔  Temporal Trend Analysis (hourly aggregation)
    ✔  Sensor Performance      (5 multi-satellite sensors)
""")

print("=" * 60)
print("  SAT-CEP-Monitor BDA Mini Project — Complete ✔")
print("  Reference: Combining CEP with Remote Sensing for NRT Air")
print("             Quality Monitoring (Morocco + Spain use case)")
print("=" * 60)

spark.stop()
print("\n✔ Spark session stopped.")

  FINAL DASHBOARD — SAT-CEP-Monitor Results

  Total Readings Processed     : 500
  Clean / Normal Readings      : 417
  Total Pollution Events (CEP) : 83
    ↳ Critical Events          : 83
    ↳ High Alert Events        : 0
    ↳ Compound Multi-Pollutant : 83

  Global Average AQI           : 48.45
  Most Polluted Region         : Madrid
  Cleanest Region              : Rabat

  Architecture Used:
    ✔  PySpark DataFrame API   (preprocessing, AQI computation)
    ✔  PySpark RDD API         (map, reduceByKey, filter, flatMap)
    ✔  Window Functions        (rolling trend / sustained pollution)
    ✔  CEP Engine              (8 rules, multi-parameter patterns)
    ✔  Ground Truth Validation (satellite vs ground station)
    ✔  Temporal Trend Analysis (hourly aggregation)
    ✔  Sensor Performance      (5 multi-satellite sensors)

  SAT-CEP-Monitor BDA Mini Project — Complete ✔
  Reference: Combining CEP with Remote Sensing for NRT Air
             Quality Monitoring (Morocco + Spain u